In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/UNSW_Blanced')

In [3]:
!pip install adversarial-robustness-toolbox


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.8 MB/s eta 0:00:00


In [4]:
#@title Env & setup
!pip -q install numpy pandas scikit-learn scipy

import os, time, random, math, json
import numpy as np
import pandas as pd
from math import ceil
from typing import Dict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score

from scipy.stats import norm
from scipy.stats import beta as beta_dist

# ------------------ Repro / Device ------------------
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


device(type='cuda')

In [5]:


#@title Load UNSW dataset → tensors
import pandas as pd
import numpy as np
import torch

train = pd.read_csv('unsw_multi_train_processed.csv')
test = pd.read_csv('unsw_multi_test_processed.csv')

X_train = train.drop(columns=['Label']).values
y_train = train['Label'].values
X_test  = test.drop(columns=['Label']).values
y_test  = test['Label'].values

X_train_tensor = torch.tensor(np.array(X_train), dtype=torch.float32)
y_train_tensor = torch.tensor(np.array(y_train), dtype=torch.long)
X_test_tensor  = torch.tensor(np.array(X_test),  dtype=torch.float32)
y_test_tensor  = torch.tensor(np.array(y_test),  dtype=torch.long)

print("Train:", X_train_tensor.shape, y_train_tensor.shape)
print("Test :", X_test_tensor.shape,  y_test_tensor.shape)




Train: torch.Size([175341, 194]) torch.Size([175341])
Test : torch.Size([82332, 194]) torch.Size([82332])


In [6]:
#@title Base model
import torch.nn as nn

class TabularNN(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        return self.fc4(x)


In [7]:
num_classes = 10
model = TabularNN(input_size=X_train_tensor.shape[1], num_classes=num_classes).to(device)

In [8]:
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test, dtype=torch.long)

In [9]:
#@title Train base model (clean)
input_size = X_train_tensor.shape[1]
num_classes = len(torch.unique(y_train_tensor))

model = TabularNN(input_size=input_size, num_classes=num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_tensor,  y_test_tensor),  batch_size=256, shuffle=False)

results_file = "results_baseline/url_evaluation_results.txt"
os.makedirs(os.path.dirname(results_file), exist_ok=True)

with open(results_file, "w") as f:
    f.write("Model Evaluation Results\n" + "="*50 + "\n")

epochs = 100
for epoch in range(epochs):
    model.train()
    run = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        run += float(loss.item())

    ep_loss = run / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {ep_loss:.4f}")
    with open(results_file, "a") as f:
        f.write(f"Epoch [{epoch+1}/{epochs}] Loss: {ep_loss:.4f}\n")

# Clean test accuracy
model.eval()
with torch.no_grad():
    logits = model(X_test_tensor.to(device))
    pred = torch.argmax(logits, dim=1).cpu()

clean_acc = accuracy_score(y_test_tensor.cpu().numpy(), pred.numpy())
print(f"Clean test accuracy: {clean_acc:.4f}")

with open(results_file, "a") as f:
    f.write(f"\nClean test accuracy: {clean_acc:.4f}\n")

Epoch [1/100] Loss: 0.6431
Epoch [2/100] Loss: 0.5576
Epoch [3/100] Loss: 0.5383
Epoch [4/100] Loss: 0.5235
Epoch [5/100] Loss: 0.5140
Epoch [6/100] Loss: 0.5085
Epoch [7/100] Loss: 0.5044
Epoch [8/100] Loss: 0.5007
Epoch [9/100] Loss: 0.4982
Epoch [10/100] Loss: 0.4964
Epoch [11/100] Loss: 0.4934
Epoch [12/100] Loss: 0.4914
Epoch [13/100] Loss: 0.4888
Epoch [14/100] Loss: 0.4871
Epoch [15/100] Loss: 0.4852
Epoch [16/100] Loss: 0.4834
Epoch [17/100] Loss: 0.4809
Epoch [18/100] Loss: 0.4801
Epoch [19/100] Loss: 0.4796
Epoch [20/100] Loss: 0.4779
Epoch [21/100] Loss: 0.4752
Epoch [22/100] Loss: 0.4756
Epoch [23/100] Loss: 0.4753
Epoch [24/100] Loss: 0.4732
Epoch [25/100] Loss: 0.4733
Epoch [26/100] Loss: 0.4726
Epoch [27/100] Loss: 0.4717
Epoch [28/100] Loss: 0.4697
Epoch [29/100] Loss: 0.4699
Epoch [30/100] Loss: 0.4696
Epoch [31/100] Loss: 0.4687
Epoch [32/100] Loss: 0.4677
Epoch [33/100] Loss: 0.4671
Epoch [34/100] Loss: 0.4661
Epoch [35/100] Loss: 0.4657
Epoch [36/100] Loss: 0.4653
E

In [ ]:
print("num classes:", len(np.unique(y_train)))
print("train classes:", np.unique(y_train, return_counts=True))
print("test classes:", np.unique(y_test, return_counts=True))

num classes: 10
train classes: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([ 2000,  1746, 12264, 33393, 18184, 40000, 56000, 10491,  1133,
         130]))
test classes: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]), array([  677,   583,  4089, 11132,  6062, 18871, 37000,  3496,   378,
          44]))


In [ ]:
from collections import Counter
print(Counter(y_train))

Counter({np.int64(6): 56000, np.int64(5): 40000, np.int64(3): 33393, np.int64(4): 18184, np.int64(2): 12264, np.int64(7): 10491, np.int64(0): 2000, np.int64(1): 1746, np.int64(8): 1133, np.int64(9): 130})


In [ ]:
print("num_classes:", len(torch.unique(y_train_tensor)))
print("train shape:", X_train_tensor.shape)
print("test shape:", X_test_tensor.shape)
print("train label counts:", np.bincount(y_train_tensor.cpu().numpy()))
print("test label counts:", np.bincount(y_test_tensor.cpu().numpy()))
print("feature min/max:", X_train_tensor.min().item(), X_train_tensor.max().item())

num_classes: 10
train shape: torch.Size([175341, 194])
test shape: torch.Size([82332, 194])
train label counts: [ 2000  1746 12264 33393 18184 40000 56000 10491  1133   130]
test label counts: [  677   583  4089 11132  6062 18871 37000  3496   378    44]
feature min/max: 0.0 1.0


In [ ]:
print(X_train_tensor.min().item(), X_train_tensor.max().item())

0.0 1.0


In [ ]:
from art.attacks.evasion import ProjectedGradientDescent, CarliniL2Method, ZooAttack, HopSkipJump, FastGradientMethod, DeepFool, SaliencyMapMethod
from art.estimators.classification import PyTorchClassifier
from torch.utils.data import DataLoader, TensorDataset
import torch
import numpy as np

# Ensure model and data are on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Create a PyTorchClassifier for ART
classifier = PyTorchClassifier(
     model=model,
     clip_values=(0, 1),
     loss=criterion,
     optimizer=optimizer,
     input_shape=(X_test_tensor.shape[1],),
     nb_classes=num_classes,
     device_type="gpu" if torch.cuda.is_available() else "cpu",
 )

# Define attack parameters
zoo_attack = ZooAttack(
     classifier,
     confidence=0.8,  # Confidence of adversarial examples
     targeted=False,
     learning_rate=0.01,
     nb_parallel= 5,
     variable_h = 0.8,
     max_iter=40,
 )

cw_attack = CarliniL2Method(
     classifier=classifier,
     confidence=0.0,
     targeted=False,
     learning_rate=0.01,
     max_iter=10,
     binary_search_steps=10,
     initial_const=0.01,
 )

pgd = ProjectedGradientDescent(estimator=classifier, eps=0.15, eps_step=0.01, max_iter=40)

deepfool = DeepFool(classifier, max_iter=10)

saliency = SaliencyMapMethod(classifier)

fgsm = FastGradientMethod(classifier, eps=0.15)

hsj = HopSkipJump(classifier, max_iter=64, max_eval=10000, init_eval=100, init_size=100)

# Convert data to numpy
X_test_np = X_test_tensor.cpu().numpy()
y_test_np = y_test_tensor.cpu().numpy()


In [ ]:
#Generate PGD adversarial examples
print("Generating PGD adversarial examples...")
X_adv_pgd = pgd.generate(X_test_np)
np.savetxt("X_adv_pgd_unsw2_brmulti.txt", X_adv_pgd, delimiter=",")
print("Saved PGD adversarial examples to 'X_adv_pgd_unsw_brmulti.txt'.")

# Generate FGSM adversarial examples
print("Generating FGSM adversarial examples...")
X_adv_fgsm = fgsm.generate(X_test_np)
np.savetxt("X_adv_fgsm_unsw2_brmulti.txt", X_adv_fgsm, delimiter=",")
print("Saved FGSM adversarial examples to 'X_adv_fgsm_unsw_brmulti.txt'.")

Generating PGD adversarial examples...


PGD - Batches:   0%|          | 0/2573 [00:00<?, ?it/s]

Saved PGD adversarial examples to 'X_adv_pgd_unsw_brmulti.txt'.
Generating FGSM adversarial examples...
Saved FGSM adversarial examples to 'X_adv_fgsm_unsw_brmulti.txt'.


In [ ]:
# Generate DeepFool adversarial examples
print("Generating DeepFool adversarial examples...")
X_adv_df = deepfool.generate(X_test_np)
np.savetxt("X_adv_df_unsw_brmulti.txt", X_adv_df, delimiter=",")
print("Saved DeepFool adversarial examples to 'X_adv_df_unsw_brmulti.txt'.")

# Generate SaliencyMapMethod adversarial examples
# print("Generating SaliencyMapMethod adversarial examples...")
# X_adv_jsma = saliency.generate(X_test_np)
# np.savetxt("X_adv_jsma_unsw_multi.txt", X_adv_jsma, delimiter=",")
# print("Saved JSMA adversarial examples to 'X_adv_jsma_unsw_multi.txt'.")

Generating DeepFool adversarial examples...


DeepFool:   0%|          | 0/82332 [00:00<?, ?it/s]

Saved DeepFool adversarial examples to 'X_adv_df_unsw_brmulti.txt'.


In [ ]:
# Generate CW adversarial examples
print("Generating CW adversarial examples...")
X_adv_cw = cw_attack.generate(X_test_np)
np.savetxt("X_adv_cw_agg_unsw2_brmulti.txt", X_adv_cw, delimiter=",")
print("Saved CW adversarial examples to 'X_adv_cw_new_brmulti.txt'.")

Generating CW adversarial examples...


C&W L_2:   0%|          | 0/82332 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Generate ZOO adversarial examples
print("Generating ZOO adversarial examples...")
X_adv_zoo = zoo_attack.generate(X_test_np)
np.savetxt("X_adv_zoo_agg_unsw_brmulti.txt", X_adv_zoo, delimiter=",")
print("Saved ZOO adversarial examples to 'X_adv_zoo_new_brmulti.txt'.")

In [ ]:
# Generate HSJ adversarial examples
print("Generating HSJ adversarial examples...")
X_adv_hsj = HopSkipJump.generate(X_test_np)
np.savetxt("X_adv_hsj_unsw_multi.txt", X_adv_hsj, delimiter=",")
print("Saved HSJ adversarial examples to 'X_adv_hsj_unsw_multi.txt'.")


In [ ]:
import torch.nn.functional as F

# Clipping function
def clip(current, low_bound, up_bound):
    # Convert bounds to tensors
    low_bound = torch.tensor(low_bound, dtype=torch.float32, device=current.device).unsqueeze(0)  # Add batch dimension
    up_bound = torch.tensor(up_bound, dtype=torch.float32, device=current.device).unsqueeze(0)    # Add batch dimension

    # Clip values
    clipped = torch.max(torch.min(current, up_bound), low_bound)
    return clipped

In [ ]:
import torch

def lowProFool(x, model, weights, bounds, maxiters, alpha, lambda_, device):
    """
    LowProFool adversarial attack implementation with weighted loss.

    Args:
        x (torch.Tensor): Input sample.
        model (torch.nn.Module): Target model.
        weights (list or np.array): Feature importance weights.
        bounds (tuple): Feature bounds as (min, max).
        maxiters (int): Maximum number of iterations.
        alpha (float): Step size for gradient update.
        lambda_ (float): Regularization weight for L2 loss.
        device (torch.device): Device to run computations on.

    Returns:
        orig_pred (int): Original prediction class.
        output_pred (int): Adversarial prediction class.
        best_pert_x (np.array): Adversarial example.
    """
    # Initialize inputs and parameters
    x = x.unsqueeze(0).to(device).float()
    x.requires_grad = True
    r = torch.tensor(1e-4 * torch.ones_like(x), device=device, requires_grad=True)  # Perturbation
    v = torch.tensor(weights, device=device).float()  # Feature importance weights

    # Initial prediction
    logits = model(x + r)
    orig_pred = logits.argmax(dim=1).item()

    target_pred = 1 - orig_pred
    target = torch.tensor([target_pred], device=device).long()

    # Define loss functions
    criterion = torch.nn.CrossEntropyLoss()
    l2 = lambda v, r: torch.sqrt(torch.sum((v * r) ** 2))

    best_norm_weighted = float('inf')
    best_pert_x = x.clone()

    # Iterative attack
    for loop_i in range(maxiters):
        if r.grad is not None:
            r.grad.zero_()

        # Compute logits and losses
        logits = model(x + r)
        loss_1 = criterion(logits, target)
        loss_2 = l2(v, r)
        loss = loss_1 + lambda_ * loss_2

        # Backpropagate the loss
        loss.backward()

        # Gradient update for perturbation
        with torch.no_grad():
            r -= alpha * r.grad  # Update perturbation

        # Clamp the perturbed sample within the bounds
        xprime = x + r
        bounds_min = torch.tensor(bounds[0], device=device).float()
        bounds_max = torch.tensor(bounds[1], device=device).float()
        xprime = torch.clamp(xprime, bounds_min, bounds_max)

        # Evaluate adversarial prediction
        logits = model(xprime)
        output_pred = logits.argmax(dim=1).item()

        # Update best adversarial example based on weighted L2 norm
        weighted_norm = torch.sum(torch.abs(r * v)).item()
        if output_pred != orig_pred and weighted_norm < best_norm_weighted:
            best_norm_weighted = weighted_norm
            best_pert_x = xprime.clone()

    # Ensure the final adversarial example is clamped
    best_pert_x = torch.clamp(best_pert_x, bounds_min, bounds_max)
    logits = model(best_pert_x)
    output_pred = logits.argmax(dim=1).item()

    return orig_pred, output_pred, best_pert_x.squeeze(0).detach().cpu().numpy()


In [ ]:
import numpy as np
import torch
from torch.autograd import Variable
from scipy.stats import pearsonr

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')



# Adjust feature importance computation to avoid constant features
feature_importance_weights = []
for i in range(X_train.shape[1]):
    if np.all(X_train[:, i] == X_train[0, i]):
        feature_importance_weights.append(0.0)
    else:
        correlation, _ = pearsonr(X_train[:, i], y_train)
        feature_importance_weights.append(abs(correlation))

feature_importance_weights = np.array(feature_importance_weights, dtype=np.float32)


feature_min = X_train.min(axis=0)
feature_max = X_train.max(axis=0)
feature_bounds = (feature_min, feature_max)

# Parameters for LowProFool attack
max_iters = 100
alpha = 0.01
lambda_ = 0.5

# Store adversarial examples
adversarial_examples = []
adversarial_accuracy = 0
num_samples = X_test.shape[0]

model.eval()
for i in range(num_samples):
    #print(f"Processing test sample {i + 1}/{num_samples}...")


    x_test_sample = X_test[i]


    x_test_sample_tensor = torch.tensor(x_test_sample, dtype=torch.float32).to(device)

    # Generate adversarial example using LowProFool attack
    orig_pred, adv_pred, adv_example = lowProFool(
        x_test_sample_tensor, model, feature_importance_weights, feature_bounds, max_iters, alpha, lambda_, device
    )


    adversarial_examples.append(adv_example)


    if orig_pred == adv_pred:
        adversarial_accuracy += 1


adversarial_examples = np.array(adversarial_examples)


np.savetxt("X_adv_lowprofool_brmulti.csv", adversarial_examples, delimiter=",")

# Calculate adversarial accuracy
adversarial_accuracy /= num_samples
print(f'Adversarial Accuracy: {adversarial_accuracy * 100:.2f}%')


In [10]:
#@title Evaluate base model on adversarial matrices (pre-defense)
def robust_load_matrix(path: str) -> np.ndarray:
    try:
        return np.loadtxt(path, delimiter=",", dtype=np.float32)
    except Exception:
        return np.loadtxt(path, dtype=np.float32)

# Example loaders (URL). Adjust paths if needed.
adversarial_loaders = {
    #"PGD": "X_adv_pgd_unsw2_brmulti.txt",
    #"JSMA": "X_adv_jsma_unsw_multi.txt",
    #"ZOO": "X_adv_zoo_agg_unsw_brmulti.txt",
    "DeepFool": "X_adv_df_unsw_brmulti.txt",
    "DeepFoolC": "X_adv_df_unsw_brmulti_c.txt",
    "ZOOC": "X_adv_zoo_agg_unsw_brmulti_c.txt",
    "PGDC": "X_adv_pgd_unsw2_brmulti_c.txt",
    "FGSMC": "X_adv_fgsm_unsw2_brmulti_c.txt",
    "JSMAC": "X_adv_jsma_unsw_multi_c.txt",
    #"FGSM": "X_adv_fgsm_unsw2_brmulti.txt",
    #"CW": "X_adv_cw_agg_unsw.txt",
    #"LPF": "X_adv_lpf_unsw.csv",
    #"HSJ": "X_adv_hsj_unsw.txt",
}

for attack_name, file_path in adversarial_loaders.items():
    X_adv = robust_load_matrix(file_path)

    if X_adv.shape[1] != X_train_tensor.shape[1]:
        print(f"[SKIP] {attack_name}: {X_adv.shape[1]} features != {X_train_tensor.shape[1]}")
        continue

    X_adv_tensor = torch.tensor(X_adv, dtype=torch.float32).to(device)
    t0 = time.time()
    with torch.no_grad():
        adv_pred = model(X_adv_tensor).argmax(1).cpu().numpy()
    dt = time.time() - t0

    adv_acc = accuracy_score(y_test_tensor.numpy(), adv_pred)
    print(f"{attack_name} adversarial accuracy: {adv_acc:.4f} | time {dt:.2f}s")
    with open(results_file, "a") as f:
        f.write(f"{attack_name} adversarial accuracy: {adv_acc:.4f}\n")
        f.write(f"Time taken for {attack_name}: {dt:.2f}s\n")


DeepFool adversarial accuracy: 0.5541 | time 0.00s
DeepFoolC adversarial accuracy: 0.6973 | time 0.00s
ZOOC adversarial accuracy: 0.7461 | time 0.00s
PGDC adversarial accuracy: 0.4925 | time 0.00s
FGSMC adversarial accuracy: 0.3657 | time 0.00s
JSMAC adversarial accuracy: 0.3325 | time 0.00s


In [11]:
###########ADAPTIVE MODEL##############


import os
import time
import numpy as np
from math import ceil
from scipy.stats import norm, beta as beta_dist

class AdaptiveSmoothEntropy:
    ABSTAIN = -1

    def __init__(self, base_classifier, num_classes, initial_sigma=1.0):
        self.base_classifier = base_classifier.to(device)
        self.num_classes = int(num_classes)
        self.sigma = float(initial_sigma)

    # ----------------------- TRAINING -----------------------

    def train_model(
        self,
        train_loader,
        test_loader,
        num_epochs=30,
        switch_epoch=None,
        ramp_epochs=5,
        lr_init=1e-3,
        lr_drop=0.5,
        beta_noisy0=0.5,
        beta_noisy1=1.0,
        lambda_consistency_base=0.1,
        noise_floor=0.20
    ):
        optimizer = optim.AdamW(self.base_classifier.parameters(), lr=lr_init, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

        if switch_epoch is None:
            switch_epoch = num_epochs // 2

        def _freeze_bn_running(m):
            if isinstance(m, nn.BatchNorm1d):
                m.eval()

        for epoch in range(num_epochs):
            self.base_classifier.train()

            if epoch < switch_epoch:
                ramp = 0.0
            else:
                if epoch == switch_epoch:
                    self.base_classifier.apply(_freeze_bn_running)
                    for g in optimizer.param_groups:
                        g['lr'] *= lr_drop
                ramp = min(1.0, (epoch - switch_epoch + 1) / max(1, ramp_epochs))

            running_loss = 0.0

            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                outputs_clean = self.base_classifier(inputs)
                loss_clean = nn.CrossEntropyLoss()(outputs_clean, labels)

                if ramp > 0.0:
                    with torch.no_grad():
                        probs = torch.softmax(outputs_clean, dim=1)
                        ent = -torch.sum(probs * torch.log(probs + 1e-8), dim=1)
                    scale = 1.0 - (ent / np.log(self.num_classes))   # [0,1]
                    scale = noise_floor + (1.0 - noise_floor) * scale
                    scale = scale * ramp

                    noise = torch.randn_like(inputs) * self.sigma * scale.unsqueeze(1)
                    noisy_inputs = inputs + noise

                    outputs_noisy = self.base_classifier(noisy_inputs)
                    loss_noisy = nn.CrossEntropyLoss()(outputs_noisy, labels)

                    with torch.no_grad():
                        p_clean = torch.softmax(outputs_clean, dim=1)
                    logp_noisy = torch.log_softmax(outputs_noisy, dim=1)
                    consistency = torch.sum(
                        p_clean * (torch.log(p_clean + 1e-8) - logp_noisy), dim=1
                    ).mean()

                    loss = (
                        loss_clean
                        + (beta_noisy0 + (beta_noisy1 - beta_noisy0) * ramp) * loss_noisy
                        + (lambda_consistency_base * ramp) * consistency
                    )
                else:
                    loss = loss_clean

                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            scheduler.step()
            print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}, ramp={ramp:.2f}")
            self.evaluate(test_loader)

    def mix_clean_and_noisy(self, inputs, ramp=1.0, noise_floor=0.20):
        """Kept for compatibility; trainer now passes ramp & noise_floor."""
        clean_inputs = inputs.clone()
        with torch.no_grad():
            probs = torch.softmax(self.base_classifier(clean_inputs), dim=1)
            entropies = -torch.sum(probs * torch.log(probs + 1e-8), dim=1)
        scale = 1.0 - (entropies / np.log(self.num_classes))
        scale = noise_floor + (1.0 - noise_floor) * scale
        scale = scale * ramp
        noise = torch.randn_like(inputs) * self.sigma * scale.unsqueeze(1)
        noisy_inputs = inputs + noise
        return clean_inputs, noisy_inputs

    def evaluate(self, loader):
        self.base_classifier.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = self.base_classifier(inputs)
                predicted = torch.argmax(outputs, dim=1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        acc = 100.0 * correct / total if total > 0 else 0.0
        print(f"Accuracy on clean data: {acc:.2f}%")
        return acc

    # ------------------- CERTIFICATION UTILS -------------------

    @staticmethod
    def _cp_lower(k: int, n: int, alpha: float) -> float:
        if n == 0 or k <= 0:
            return 0.0
        return float(beta_dist.ppf(alpha, k, n - k + 1))

    @staticmethod
    def _cp_upper(k: int, n: int, alpha: float) -> float:
        if n == 0 or k >= n:
            return 1.0
        return float(beta_dist.ppf(1.0 - alpha, k + 1, n - k))

    @staticmethod
    def _ppf_safe(p: float) -> float:
        p = float(np.clip(p, 1e-12, 1 - 1e-12))
        return float(norm.ppf(p))

    def _proxy_margin(self, x):
        with torch.no_grad():
            probs = torch.softmax(self.base_classifier(x), dim=1)
        top2 = torch.topk(probs, 2).values.squeeze()
        return (top2[0] - top2[1]).item(), top2[0].item()


    def _adaptive_sample_size(self, margin, confidence, n_base, max_scale=3.0, gamma=2.0):
        scale = max(1.0, (1.0 - margin) * (1.0 - confidence) * gamma)
        scale = min(scale, max_scale)
        return int(ceil(n_base * scale))

    def _sample_noise(self, x, num, batch_size):
        counts = np.zeros(self.num_classes, dtype=int)
        remaining = int(num)
        with torch.no_grad():
            while remaining > 0:
                bs = min(batch_size, remaining)
                remaining -= bs
                batch = x.repeat((bs, 1)).float().to(device)
                noise = torch.randn_like(batch) * self.sigma
                predictions = self.base_classifier(batch + noise).argmax(1)
                counts += np.bincount(predictions.cpu().numpy(), minlength=self.num_classes)
        return counts

    # --------------------- CERTIFICATION ---------------------

    def certify(self, x, n0=100, n_base=1000, alpha=0.001, batch_size=256):
        """
        RS certification with one-sided CP bounds.
        Uses adaptive n >= n_base. Binary case gets tighter radius.
        Returns: (predicted_class, certified_radius)
        """
        self.base_classifier.eval()

        # selection
        counts_selection = self._sample_noise(x, n0, batch_size)
        cA_hat = int(np.argmax(counts_selection))

        # adapt n by margin+confidence
        margin, confidence = self._proxy_margin(x)
        n = self._adaptive_sample_size(margin, confidence, n_base=n_base)

        # estimation
        counts_estimation = self._sample_noise(x, n, batch_size)
        nA = int(counts_estimation[cA_hat])

        pA_LB = self._cp_lower(nA, n, alpha)


        if self.num_classes == 2:
            if pA_LB <= 0.5:
                return AdaptiveSmoothEntropy.ABSTAIN, 0.0
            R = self.sigma * self._ppf_safe(pA_LB)
            return cA_hat, max(0.0, R)

        # multiclass uses runner-up UB
        cB_hat = int(np.argsort(counts_estimation)[-2]) if self.num_classes > 1 else cA_hat
        nB = int(counts_estimation[cB_hat])
        pB_UB = self._cp_upper(nB, n, alpha)

        if pA_LB <= 0.5 or pA_LB <= pB_UB:
            return AdaptiveSmoothEntropy.ABSTAIN, 0.0

        R = 0.5 * self.sigma * (self._ppf_safe(pA_LB) - self._ppf_safe(pB_UB))
        return cA_hat, max(0.0, R)

    # ---------------------- ADV TESTING ----------------------

    def test_on_adversarial(self, X_adv_test, y_test, method="margin",
                            n0=100, n=1000, alpha=0.001, batch_size=64,
                            save_to: str = None, tag: str = ""):
        """
        Evaluate on an adversarial matrix (rows aligned with y_test).

        Returns:
          (accuracy_pct, certified_accuracy_pct, abstention_rate_pct,
           avg_certified_radius, total_time_seconds, certify_ms_per_sample)
        """
        total = len(X_adv_test)
        correct = 0
        certified_correct = 0
        abstain = 0
        radii = []

        t_all_start = time.time()
        certify_time = 0.0

        with torch.no_grad():
            for i in range(total):
                x_adv_sample = torch.tensor(X_adv_test[i]).unsqueeze(0).to(device).float()
                y_true = int(y_test[i].item() if hasattr(y_test[i], 'item') else y_test[i])

                t0 = time.time()
                certified_class, certified_radius = self.certify(
                    x_adv_sample, n0=n0, n_base=n, alpha=alpha, batch_size=batch_size
                )
                certify_time += (time.time() - t0)

                outputs = self.base_classifier(x_adv_sample)
                predicted = torch.argmax(outputs, dim=1).item()

                if predicted == y_true:
                    correct += 1
                if certified_class == y_true:
                    certified_correct += 1
                if certified_class == AdaptiveSmoothEntropy.ABSTAIN:
                    abstain += 1
                if certified_radius > 0:
                    radii.append(certified_radius)

        total_time = time.time() - t_all_start
        ms_per_sample = (certify_time / max(total, 1)) * 1000.0

        accuracy = (correct / total * 100.0) if total > 0 else 0.0
        certified_accuracy = (certified_correct / total * 100.0) if total > 0 else 0.0
        abstention_rate = (abstain / total * 100.0) if total > 0 else 0.0
        avg_radius = float(np.mean(radii)) if radii else 0.0

        msg = (f"{tag} {method.capitalize()} | Acc: {accuracy:.2f}% | "
               f"Certified: {certified_accuracy:.2f}% | Abstain: {abstention_rate:.2f}% | "
               f"Avg R: {avg_radius:.4f} | TimeTot: {total_time:.2f}s | Cert ms/sample: {ms_per_sample:.2f}")
        print(msg)

        if save_to is not None:
            os.makedirs(os.path.dirname(save_to), exist_ok=True) if os.path.dirname(save_to) else None
            with open(save_to, "a") as f:
                f.write(msg + "\n")


        return accuracy, certified_accuracy, abstention_rate, avg_radius, total_time, ms_per_sample

    # -------- convenience for sigma sweeps --------
    def set_sigma(self, new_sigma: float):
        self.sigma = float(new_sigma)


In [12]:
###########ADAPTIVE MODEL##############
from sklearn.model_selection import StratifiedKFold

def train_and_evaluate_on_adversarial_folds(
    base_model_class,
    adaptive_model_class,
    X_train_tensor,
    y_train_tensor,
    X_test_tensor,
    y_test_tensor,
    adversarial_loaders,
    num_classes,
    initial_sigma,
    batch_size=64,
    num_epochs=30,
    folds=5,
    output_file="results/sigma0.5_masking_less_url_adaptive_smoothing.txt",
):
    import os, time
    import numpy as np
    import torch
    from torch.utils.data import DataLoader, TensorDataset, Subset
    from sklearn.model_selection import KFold
    from collections import defaultdict

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- helpers ---
    def _ensure_results_dir(path):
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)

    def _load_matrix(path: str) -> np.ndarray:

        try:
            return np.loadtxt(path, delimiter=",", dtype=np.float32)
        except Exception:
            return np.loadtxt(path, dtype=np.float32)


    preloaded_adv = {}
    for attack_name, file_path in adversarial_loaders.items():
        if attack_name.lower() == "clean":

            preloaded_adv[attack_name] = X_test_tensor.cpu().numpy().astype(np.float32)
        else:
            mat = _load_matrix(file_path)
            preloaded_adv[attack_name] = mat.astype(np.float32)


    n_test = X_test_tensor.shape[0]
    for name, mat in preloaded_adv.items():
        if mat.shape[0] < n_test:
            raise ValueError(f"[{name}] rows {mat.shape[0]} < test set size {n_test}. "
                             "Adversarial file must align with the full test set so we can index folds.")

    # Datasets & loaders
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)


    kf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)

    y_test_np = y_test_tensor.cpu().numpy()
    #kf = KFold(n_splits=folds, shuffle=True, random_state=42)

    _ensure_results_dir(output_file)
    with open(output_file, "w") as f:
        f.write("===== Adversarial Evaluation Results =====\n")
        f.write(f"initial_sigma = {initial_sigma}\n")
        f.write(f"folds = {folds}, batch_size = {batch_size}, epochs = {num_epochs}\n\n")

    # Initialize & train
    print("Initializing models...")
    #base_model = base_model_class(input_size=X_train_tensor.shape[1], dropout_rate=0.5).to(device)
    base_model = base_model_class(input_size=X_train_tensor.shape[1], num_classes=num_classes, dropout_rate=0.5).to(device)
    adaptive_model = adaptive_model_class(
        base_classifier=base_model,
        num_classes=num_classes,
        initial_sigma=initial_sigma,
    )

    print("Training AdaptiveSmoothEntropy model...")
    adaptive_model.train_model(train_loader, test_loader, num_epochs=num_epochs)

    # Clean evaluation
    print("Evaluating on clean testing data (full):")
    clean_accuracy = adaptive_model.evaluate(test_loader)
    print(f"Clean Test Accuracy: {clean_accuracy:.2f}%")
    with open(output_file, "a") as f:
        f.write(f"Clean Test Accuracy (full): {clean_accuracy:.2f}%\n")


    adversarial_summary = defaultdict(list)

    for fold, (_, test_idx) in enumerate(kf.split(X_test_tensor, y_test_np), start=1):
        print(f"\n===== Fold {fold}/{folds} =====")
        val_subset = Subset(test_dataset, test_idx)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)


        y_val = torch.stack([val_subset[i][1] for i in range(len(val_subset))]).cpu().numpy()

        with open(output_file, "a") as f:
            f.write(f"\nFold {fold} (Clean Acc full test: {clean_accuracy:.2f}%)\n")

        for attack_name, mat in preloaded_adv.items():
            print(f"\nEvaluating {attack_name} on Fold {fold}...")
            # pick rows of this fold
            X_adv_fold = mat[np.array(test_idx)]
            X_adv_tensor = torch.tensor(X_adv_fold, dtype=torch.float32, device=device)


            acc, cert_acc, abstain_pct, avg_R, time_s, ms_ps = adaptive_model.test_on_adversarial(
                X_adv_tensor,
                y_val,
                method="margin",
                n0=200,
                n=1000,
                alpha=0.001,
                batch_size=batch_size,
                save_to=output_file,
                tag=f"[sigma={initial_sigma}] [fold={fold}] [attack={attack_name}]"
            )

            # record
            adversarial_summary[attack_name].append(
                (acc, cert_acc, abstain_pct, avg_R, time_s, ms_ps)
            )

            # per-fold log line
            with open(output_file, "a") as f:
                f.write(
                    f"  {attack_name}: "
                    f"Acc={acc:.2f}%, "
                    f"Certified={cert_acc:.2f}%, "
                    f"Abstain={abstain_pct:.2f}%, "
                    f"AvgR={avg_R:.4f}, "
                    f"TimeTot={time_s:.2f}s, "
                    f"Cert ms/sample={ms_ps:.2f}\n"
                )

    # Summary across folds
    print("\n===== Adversarial Evaluation Summary =====")
    with open(output_file, "a") as f:
        f.write("\n===== Adversarial Evaluation Summary =====\n")
        for attack_name, metrics in adversarial_summary.items():
            arr = np.array(metrics, dtype=float)  # shape [folds, 6]
            avg_acc   = float(np.mean(arr[:, 0]))
            avg_cert  = float(np.mean(arr[:, 1]))
            avg_abst  = float(np.mean(arr[:, 2]))
            avg_rad   = float(np.mean(arr[:, 3]))
            avg_time  = float(np.mean(arr[:, 4]))
            avg_msps  = float(np.mean(arr[:, 5]))
            line = (f"{attack_name}: Avg Acc={avg_acc:.2f}%, "
                    f"Avg Certified={avg_cert:.2f}%, "
                    f"Avg Abstain={avg_abst:.2f}%, "
                    f"Avg R={avg_rad:.4f}, "
                    f"Avg Time={avg_time:.2f}s, "
                    f"Avg Cert ms/sample={avg_msps:.2f}")
            print(line)
            f.write(line + "\n")

    return adversarial_summary


In [13]:
###########ADAPTIVE MODEL##############

import os
import numpy as np

adversarial_loaders = {
    #"PGD": "X_adv_pgd_unsw2_brmulti.txt",
    #"JSMA": "X_adv_jsma_unsw_multi.txt",
    #"ZOO": "X_adv_zoo_agg_unsw_brmulti.txt",
    "DeepFool": "X_adv_df_unsw_brmulti.txt",
    "DeepFoolC": "X_adv_df_unsw_brmulti_c.txt",
    "ZOOC": "X_adv_zoo_agg_unsw_brmulti_c.txt",
    "PGDC": "X_adv_pgd_unsw2_brmulti_c.txt",
    "FGSMC": "X_adv_fgsm_unsw2_brmulti_c.txt",
    "JSMAC": "X_adv_jsma_unsw_multi_c.txt",
    #"FGSM": "X_adv_fgsm_unsw2_brmulti.txt",
    #"CW": "X_adv_cw_agg_unsw.txt",
    #"LPF": "X_adv_lpf_unsw.csv",
    #"HSJ": "X_adv_hsj_unsw.txt",
}

# -------------- sweep settings --------------
sigma_grid = [0.30]
results_dir = "results_sigma_sweep"
os.makedirs(results_dir, exist_ok=True)

# Combined sweep summary (text)
combined_summary_path = os.path.join(results_dir, "sweep_summary.txt")
with open(combined_summary_path, "w") as f:
    f.write("===== Multi RS σ Sweep Summary =====\n")
    f.write(f"sigma_grid = {sigma_grid}\n\n")
    f.write("Reported metrics: Avg Acc (%), Avg Certified (%), Avg Abstain (%), "
            "Avg R, Avg Time (s), Avg Cert ms/sample\n\n")

# Helper to average metrics from the dict returned by Part 2
def _avg_metrics(summary_dict):
    """
    summary_dict: attack -> list of tuples
      (acc, cert_acc, abstain_pct, avg_R, total_time_s, ms_per_sample)
    Returns: attack -> (mean_acc, mean_cert_acc, mean_abstain, mean_avgR, mean_time_s, mean_ms_per_ex)
    """
    out = {}
    for attack, tuples in summary_dict.items():
        if len(tuples) == 0:
            continue
        arr = np.array(tuples, dtype=float)
        mean_acc       = float(np.mean(arr[:, 0]))
        mean_cert_acc  = float(np.mean(arr[:, 1]))
        mean_abstain   = float(np.mean(arr[:, 2]))
        mean_avgR      = float(np.mean(arr[:, 3]))
        mean_time_s    = float(np.mean(arr[:, 4]))
        mean_ms_per_ex = float(np.mean(arr[:, 5]))
        out[attack] = (mean_acc, mean_cert_acc, mean_abstain, mean_avgR, mean_time_s, mean_ms_per_ex)
    return out

all_sigma_results = {}

# -------------- rigorous sweep (retrain per σ) --------------
for sigma in sigma_grid:
    per_sigma_file = os.path.join(results_dir, f"sigma_{sigma:.2f}.txt")

    results = train_and_evaluate_on_adversarial_folds(
        base_model_class=TabularNN,
        adaptive_model_class=AdaptiveSmoothEntropy,
        X_train_tensor=X_train_tensor,
        y_train_tensor=y_train_tensor,
        X_test_tensor=X_test_tensor,
        y_test_tensor=y_test_tensor,
        adversarial_loaders=adversarial_loaders,
        num_classes=num_classes,               # multi
        initial_sigma=sigma,
        batch_size=64,
        num_epochs=30,
        folds=5,
        output_file=per_sigma_file,
    )

    # Average across folds for this σ and append to combined summary
    avg = _avg_metrics(results)
    all_sigma_results[sigma] = avg

    with open(combined_summary_path, "a") as f:
        f.write(f"\n=== σ = {sigma:.2f} ===\n")
        for attack in adversarial_loaders.keys():
            if attack not in avg:
                continue
            acc, cert_acc, abstain, avgR, time_s, ms_ps = avg[attack]
            f.write(
                f"{attack}: "
                f"Avg Acc={acc:.2f}%, "
                f"Avg Certified={cert_acc:.2f}%, "
                f"Avg Abstain={abstain:.2f}%, "
                f"Avg R={avgR:.4f}, "
                f"Avg Time={time_s:.2f}s, "
                f"Avg Cert ms/sample={ms_ps:.2f}\n"
            )

print(f"\nPer-σ files saved under: {results_dir}")
print(f"Combined summary saved to: {combined_summary_path}")


Initializing models...
Training AdaptiveSmoothEntropy model...
Epoch 1/30, Loss: 0.6737, ramp=0.00
Accuracy on clean data: 70.91%
Epoch 2/30, Loss: 0.5831, ramp=0.00
Accuracy on clean data: 69.95%
Epoch 3/30, Loss: 0.5681, ramp=0.00
Accuracy on clean data: 74.56%
Epoch 4/30, Loss: 0.5571, ramp=0.00
Accuracy on clean data: 74.11%
Epoch 5/30, Loss: 0.5492, ramp=0.00
Accuracy on clean data: 71.81%
Epoch 6/30, Loss: 0.5435, ramp=0.00
Accuracy on clean data: 72.39%
Epoch 7/30, Loss: 0.5382, ramp=0.00
Accuracy on clean data: 76.33%
Epoch 8/30, Loss: 0.5359, ramp=0.00
Accuracy on clean data: 75.17%
Epoch 9/30, Loss: 0.5300, ramp=0.00
Accuracy on clean data: 75.68%
Epoch 10/30, Loss: 0.5288, ramp=0.00
Accuracy on clean data: 73.47%
Epoch 11/30, Loss: 0.5168, ramp=0.00
Accuracy on clean data: 75.12%
Epoch 12/30, Loss: 0.5138, ramp=0.00
Accuracy on clean data: 75.31%
Epoch 13/30, Loss: 0.5110, ramp=0.00
Accuracy on clean data: 74.64%
Epoch 14/30, Loss: 0.5110, ramp=0.00
Accuracy on clean data: 7

/tmp/ipykernel_7280/1162896085.py:238: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x_adv_sample = torch.tensor(X_adv_test[i]).unsqueeze(0).to(device).float()


[sigma=0.3] [fold=1] [attack=DeepFool] Margin | Acc: 74.01% | Certified: 62.90% | Abstain: 8.23% | Avg R: 0.3900 | TimeTot: 178.87s | Cert ms/sample: 10.37

Evaluating DeepFoolC on Fold 1...
[sigma=0.3] [fold=1] [attack=DeepFoolC] Margin | Acc: 74.03% | Certified: 62.85% | Abstain: 8.28% | Avg R: 0.3969 | TimeTot: 181.81s | Cert ms/sample: 10.52

Evaluating ZOOC on Fold 1...
[sigma=0.3] [fold=1] [attack=ZOOC] Margin | Acc: 74.82% | Certified: 62.92% | Abstain: 8.16% | Avg R: 0.4191 | TimeTot: 181.67s | Cert ms/sample: 10.52

Evaluating PGDC on Fold 1...
[sigma=0.3] [fold=1] [attack=PGDC] Margin | Acc: 66.51% | Certified: 60.89% | Abstain: 15.80% | Avg R: 0.3899 | TimeTot: 178.85s | Cert ms/sample: 10.36

Evaluating FGSMC on Fold 1...
[sigma=0.3] [fold=1] [attack=FGSMC] Margin | Acc: 64.80% | Certified: 61.95% | Abstain: 8.40% | Avg R: 0.3542 | TimeTot: 178.97s | Cert ms/sample: 10.37

Evaluating JSMAC on Fold 1...
[sigma=0.3] [fold=1] [attack=JSMAC] Margin | Acc: 58.78% | Certified: 54

In [14]:

### code to compare with the original randomized smoothing ###

import torch
import os
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import norm
from statsmodels.stats.proportion import proportion_confint
import numpy as np
from math import ceil

# Define your Tabular Neural Network

class TabularNN(nn.Module):
    def __init__(self, input_size, num_classes, dropout_rate=0.2):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.relu(self.fc3(x))
        return self.fc4(x)

# Original Randomized Smoothing (certification method)
class Smooth:
    ABSTAIN = -1

    def __init__(self, base_classifier: nn.Module, num_classes: int, sigma: float, device=None):
        self.base_classifier = base_classifier
        self.num_classes = num_classes
        self.sigma = sigma
        if device is None:
            device = next(base_classifier.parameters()).device
        self.device = device

    def certify(self, x: torch.tensor, n0: int, n: int, alpha: float, batch_size: int):
        self.base_classifier.eval()
        x = x.to(self.device).float()

        counts_selection = self._sample_noise(x, n0, batch_size)
        cAHat = int(np.argmax(counts_selection))

        counts_estimation = self._sample_noise(x, n, batch_size)
        nA = int(counts_estimation[cAHat])
        pA_LB = self._lower_confidence_bound(nA, n, alpha)

        # binary case
        if self.num_classes == 2:
            if pA_LB <= 0.5:
                return Smooth.ABSTAIN, 0.0
            radius = self.sigma * norm.ppf(np.clip(pA_LB, 1e-12, 1 - 1e-12))
            return cAHat, max(0.0, float(radius))

        # multi-class case: use runner-up upper bound
        cBHat = int(np.argsort(counts_estimation)[-2])
        nB = int(counts_estimation[cBHat])
        pB_UB = self._upper_confidence_bound(nB, n, alpha)

        if pA_LB <= 0.5 or pA_LB <= pB_UB:
            return Smooth.ABSTAIN, 0.0

        zA = norm.ppf(np.clip(pA_LB, 1e-12, 1 - 1e-12))
        zB = norm.ppf(np.clip(pB_UB, 1e-12, 1 - 1e-12))
        radius = 0.5 * self.sigma * (zA - zB)
        return cAHat, max(0.0, float(radius))

    def _sample_noise(self, x: torch.tensor, num: int, batch_size: int):
        with torch.no_grad():
            counts = np.zeros(self.num_classes, dtype=int)
            remaining = num
            while remaining > 0:
                this_batch_size = min(batch_size, remaining)
                remaining -= this_batch_size
                batch = x.repeat((this_batch_size, 1)).to(self.device)
                noise = torch.randn_like(batch) * self.sigma
                predictions = self.base_classifier(batch + noise).argmax(1)
                counts += np.bincount(predictions.cpu().numpy(), minlength=self.num_classes)
            return counts

    def _lower_confidence_bound(self, NA: int, N: int, alpha: float) -> float:
        return proportion_confint(NA, N, alpha=2 * alpha, method="beta")[0]

    def _upper_confidence_bound(self, NB: int, N: int, alpha: float) -> float:
        return proportion_confint(NB, N, alpha=2 * alpha, method="beta")[1]
# Training loop with noise
def train_model(model, train_loader, criterion, optimizer, noise_sd, num_epochs=10, device=None):
    if device is None:
        device = next(model.parameters()).device
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            noisy_inputs = inputs + torch.randn_like(inputs) * noise_sd
            outputs = model(noisy_inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss / len(train_loader):.4f}")

# Evaluation function
def evaluate_model(model, test_loader, noise_sd, device=None):
    if device is None:
        device = next(model.parameters()).device
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            noisy_inputs = inputs + torch.randn_like(inputs) * noise_sd
            outputs = model(noisy_inputs)
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

# ----evaluation against adversarial inputs with timing ----
def test_on_adversarial_original(
    smoother,
    X_adv_tensor,
    y_true_np,
    n0=100, n=1000, alpha=0.001, batch_size=64,
    save_to: str = None, tag: str = ""
):
    import time
    device = next(smoother.base_classifier.parameters()).device

    total = X_adv_tensor.size(0)
    correct = 0
    certified_correct = 0
    abstain = 0
    radii = []

    t_all_start = time.time()
    certify_time = 0.0

    with torch.no_grad():
        for i in range(total):
            x_adv = X_adv_tensor[i].unsqueeze(0).to(device)
            y_true = int(y_true_np[i])

            t0 = time.time()
            c_hat, r = smoother.certify(
                x_adv, n0=n0, n=n, alpha=alpha, batch_size=batch_size
            )
            certify_time += (time.time() - t0)

            # plain prediction (no noise)
            logits = smoother.base_classifier(x_adv)
            y_pred = int(torch.argmax(logits, dim=1).item())

            if y_pred == y_true:
                correct += 1
            if c_hat == y_true:
                certified_correct += 1
            if c_hat == smoother.ABSTAIN:
                abstain += 1
            if r > 0:
                radii.append(float(r))

    total_time = time.time() - t_all_start
    ms_per_sample = (certify_time / max(total, 1)) * 1000.0

    acc = 100.0 * correct / total if total > 0 else 0.0
    cert_acc = 100.0 * certified_correct / total if total > 0 else 0.0
    abstain_pct = 100.0 * abstain / total if total > 0 else 0.0
    avg_r = float(np.mean(radii)) if len(radii) > 0 else 0.0

    msg = (f"{tag} OriginalRS | Acc: {acc:.2f}% | "
           f"Certified: {cert_acc:.2f}% | Abstain: {abstain_pct:.2f}% | "
           f"Avg R: {avg_r:.4f} | TimeTot: {total_time:.2f}s | Cert ms/sample: {ms_per_sample:.2f}")
    print(msg)
    if save_to is not None:
        os.makedirs(os.path.dirname(save_to), exist_ok=True) if os.path.dirname(save_to) else None
        with open(save_to, "a") as f:
            f.write(msg + "\n")

    # Return 6-tuple to match your ASE pipeline
    return acc, cert_acc, abstain_pct, avg_r, total_time, ms_per_sample


In [15]:
from sklearn.model_selection import StratifiedKFold


def train_and_evaluate_on_adversarial_folds(
    base_model_class,
    adaptive_model_class,
    X_train_tensor,
    y_train_tensor,
    X_test_tensor,
    y_test_tensor,
    adversarial_loaders,
    num_classes,
    initial_sigma,
    batch_size=64,
    num_epochs=30,
    folds=5,
    output_file="unsw_originalSmoothing.txt",
):
    import os, time
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset, Subset
    from sklearn.model_selection import KFold
    from collections import defaultdict

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ---------- helpers ----------
    def _load_matrix(path: str) -> np.ndarray:
        try:
            return np.loadtxt(path, delimiter=",", dtype=np.float32)
        except Exception:
            return np.loadtxt(path, dtype=np.float32)

    def _ensure_dir(path: str):
        d = os.path.dirname(path)
        if d:
            os.makedirs(d, exist_ok=True)

    # ---------- file header ----------
    _ensure_dir(output_file)
    with open(output_file, "w") as f:
        f.write("===== Adversarial Evaluation Results (Original RS) =====\n")
        f.write(f"initial_sigma = {initial_sigma}\n")
        f.write(f"folds        = {folds}\n")
        f.write(f"batch_size   = {batch_size}\n")
        f.write(f"epochs       = {num_epochs}\n")
        f.write("Reported metrics: Acc (%), Certified (%), Abstain (%), Avg R, "
                "TimeTot (s), Cert ms/sample\n\n")

    # ---------- datasets & loaders ----------
    test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader   = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    # ---------- preload adversarial matrices ----------
    preloaded_adv = {}
    for name, path in adversarial_loaders.items():
        mat = _load_matrix(path).astype(np.float32)
        preloaded_adv[name] = mat

    n_test = X_test_tensor.shape[0]
    for name, mat in preloaded_adv.items():
        if mat.shape[0] < n_test:
            raise ValueError(f"[{name}] has {mat.shape[0]} rows < test size {n_test}.")
        if mat.shape[1] != X_train_tensor.shape[1]:
            raise ValueError(f"[{name}] feature dim {mat.shape[1]} != train dim {X_train_tensor.shape[1]}.")

    # ---------- init base + smoother ----------
    print("Initializing models...")
    base_model = base_model_class(input_size=X_train_tensor.shape[1],num_classes=num_classes,dropout_rate=0.5).to(device)
    #base_model = base_model_class(input_size=X_train_tensor.shape[1], dropout_rate=0.5).to(device)
    smoother = adaptive_model_class(base_classifier=base_model, num_classes=num_classes, sigma=float(initial_sigma))

    # ---------- train base (with Gaussian noise_sd = initial_sigma) ----------
    print("Training OriginalSmooth model...")
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(base_model.parameters(), lr=1e-3)
    noise_sd = float(initial_sigma)   # match σ

    base_model.train()
    for epoch in range(num_epochs):
        running = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device); yb = yb.to(device)
            optimizer.zero_grad()
            noisy_xb = xb + torch.randn_like(xb) * noise_sd
            logits = base_model(noisy_xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            running += loss.item()
        print(f"[OriginalRS] Epoch {epoch+1}/{num_epochs} loss={running/len(train_loader):.4f}")

    # ---------- clean eval ----------
    print("Evaluating OriginalSmooth model on clean test...")
    base_model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = base_model(xb).argmax(1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)
    clean_acc = 100.0 * correct / total if total > 0 else 0.0
    print(f"Clean Test Accuracy: {clean_acc:.2f}%")
    with open(output_file, "a") as f:
        f.write(f"Clean Test Accuracy: {clean_acc:.2f}%\n")

    # ---------- k-fold on adversarial ----------
    kf = StratifiedKFold(n_splits=folds, shuffle=True, random_state=42)
    y_test_np = y_test_tensor.cpu().numpy()

    adversarial_summary = defaultdict(list)

    for fold, (_, test_idx) in enumerate(kf.split(X_test_tensor, y_test_np), start=1):
        print(f"\n===== Fold {fold}/{folds} =====")
        val_subset = Subset(test_dataset, test_idx)
        y_val = torch.stack([val_subset[i][1] for i in range(len(val_subset))]).cpu().numpy()

        with open(output_file, "a") as f:
            f.write(f"\nFold {fold} (Clean Acc full test: {clean_acc:.2f}%)\n")

        for attack_name, mat in preloaded_adv.items():
            X_adv_fold = torch.tensor(mat[np.array(test_idx)], dtype=torch.float32, device=device)

            acc, cert_acc, abstain_pct, avg_r, time_s, ms_ps = test_on_adversarial_original(
                smoother, X_adv_fold, y_val,
                n0=100, n=1000, alpha=0.001, batch_size=batch_size,
                save_to=output_file,
                tag=f"[sigma={initial_sigma}] [fold={fold}] [attack={attack_name}]"
            )

            adversarial_summary[attack_name].append((acc, cert_acc, abstain_pct, avg_r, time_s, ms_ps))

            # per-fold log line
            with open(output_file, "a") as f:
                f.write(
                    f"  {attack_name}: "
                    f"Acc={acc:.2f}%, "
                    f"Certified={cert_acc:.2f}%, "
                    f"Abstain={abstain_pct:.2f}%, "
                    f"AvgR={avg_r:.4f}, "
                    f"TimeTot={time_s:.2f}s, "
                    f"Cert ms/sample={ms_ps:.2f}\n"
                )

    # ---------- summary across folds ----------
    print("\n===== Adversarial Evaluation Summary =====")
    with open(output_file, "a") as f:
        f.write("\n===== Adversarial Evaluation Summary =====\n")
        for atk, metrics in adversarial_summary.items():
            arr = np.array(metrics, dtype=float)  # [folds, 6]
            avg_acc   = float(arr[:, 0].mean())
            avg_cert  = float(arr[:, 1].mean())
            avg_abs   = float(arr[:, 2].mean())
            avg_r     = float(arr[:, 3].mean())
            avg_time  = float(arr[:, 4].mean())
            avg_msps  = float(arr[:, 5].mean())
            line = (f"{atk}: Avg Acc={avg_acc:.2f}%, "
                    f"Avg Certified={avg_cert:.2f}%, "
                    f"Avg Abstain={avg_abs:.2f}%, "
                    f"Avg R={avg_r:.4f}, "
                    f"Avg Time={avg_time:.2f}s, "
                    f"Avg Cert ms/sample={avg_msps:.2f}")
            print(line); f.write(line + "\n")

    return adversarial_summary


In [ ]:
# ===== σ-sweep for Original Randomized Smoothing  =====
import os
import numpy as np
import torch

adversarial_loaders = {
    #"PGD": "X_adv_pgd_unsw2_brmulti.txt",
    #"JSMA": "X_adv_jsma_unsw_multi.txt",
    #"ZOO": "X_adv_zoo_agg_unsw_brmulti.txt",
    "DeepFool": "X_adv_df_unsw_brmulti.txt",
    "DeepFoolC": "X_adv_df_unsw_brmulti_c.txt",
    "ZOOC": "X_adv_zoo_agg_unsw_brmulti_c.txt",
    "PGDC": "X_adv_pgd_unsw2_brmulti_c.txt",
    "FGSMC": "X_adv_fgsm_unsw2_brmulti_c.txt",
    "JSMAC": "X_adv_jsma_unsw_multi_c.txt",
    #"FGSM": "X_adv_fgsm_unsw2_brmulti.txt",
    #"CW": "X_adv_cw_agg_unsw.txt",
    #"LPF": "X_adv_lpf_unsw.csv",
    #"HSJ": "X_adv_hsj_unsw.txt",
}


# ---- sweep settings ----
sigma_grid = [0.3]
results_dir = "results_original_rs_sigma_sweep"
os.makedirs(results_dir, exist_ok=True)

num_classes = int(y_train_tensor.max().item()) + 1


combined_summary_path = os.path.join(results_dir, "sweep_summary_original_rs.txt")
with open(combined_summary_path, "w") as f:
    f.write("===== Original RS σ Sweep Summary (Multi-class) =====\n")
    f.write(f"sigma_grid = {sigma_grid}\n")
    f.write(f"num_classes = {num_classes}\n\n")
    f.write("Reported metrics: Avg Acc (%), Avg Certified (%), Avg Abstain (%), "
            "Avg R, Avg Time (s), Avg Cert ms/sample\n\n")

def _avg_metrics(summary_dict):
    """
    summary_dict: {attack: [(acc, cert_acc, abstain_pct, avg_R, total_time_s, ms_per_sample), ... per fold]}
    returns: {attack: (mean_acc, mean_cert, mean_abstain, mean_avgR, mean_time_s, mean_ms_per_ex)}
    """
    out = {}
    for attack, tuples in summary_dict.items():
        if len(tuples) == 0:
            continue
        arr = np.array(tuples, dtype=float)
        mean_acc       = float(arr[:, 0].mean())
        mean_cert      = float(arr[:, 1].mean())
        mean_abstain   = float(arr[:, 2].mean())
        mean_avgR      = float(arr[:, 3].mean())
        mean_time_s    = float(arr[:, 4].mean())
        mean_ms_per_ex = float(arr[:, 5].mean())
        out[attack] = (mean_acc, mean_cert, mean_abstain, mean_avgR, mean_time_s, mean_ms_per_ex)
    return out

all_sigma_results = {}

for sigma in sigma_grid:
    per_sigma_file = os.path.join(results_dir, f"orig_rs_sigma_{sigma:.2f}.txt")

    results = train_and_evaluate_on_adversarial_folds(
        base_model_class=TabularNN,
        adaptive_model_class=Smooth,
        X_train_tensor=X_train_tensor,
        y_train_tensor=y_train_tensor,
        X_test_tensor=X_test_tensor,
        y_test_tensor=y_test_tensor,
        adversarial_loaders=adversarial_loaders,
        num_classes=num_classes,   # multi-class fix
        initial_sigma=sigma,
        batch_size=64,
        num_epochs=30,
        folds=5,
        output_file=per_sigma_file,
    )

    avg = _avg_metrics(results)
    all_sigma_results[sigma] = avg

    with open(combined_summary_path, "a") as f:
        f.write(f"\n=== σ = {sigma:.2f} ===\n")
        for attack in adversarial_loaders.keys():
            if attack not in avg:
                continue
            acc, cert_acc, abstain, avgR, time_s, ms_ps = avg[attack]
            f.write(
                f"{attack}: "
                f"Avg Acc={acc:.2f}%, "
                f"Avg Certified={cert_acc:.2f}%, "
                f"Avg Abstain={abstain:.2f}%, "
                f"Avg R={avgR:.4f}, "
                f"Avg Time={time_s:.2f}s, "
                f"Avg Cert ms/sample={ms_ps:.2f}\n"
            )

print(f"\nPer-σ files saved under: {results_dir}")
print(f"Combined summary saved to: {combined_summary_path}")

Initializing models...
Training OriginalSmooth model...
[OriginalRS] Epoch 1/30 loss=0.9084
[OriginalRS] Epoch 2/30 loss=0.8170
[OriginalRS] Epoch 3/30 loss=0.8032
[OriginalRS] Epoch 4/30 loss=0.7958
[OriginalRS] Epoch 5/30 loss=0.7901
[OriginalRS] Epoch 6/30 loss=0.7885
[OriginalRS] Epoch 7/30 loss=0.7847
[OriginalRS] Epoch 8/30 loss=0.7836
[OriginalRS] Epoch 9/30 loss=0.7826
[OriginalRS] Epoch 10/30 loss=0.7798
[OriginalRS] Epoch 11/30 loss=0.7804
[OriginalRS] Epoch 12/30 loss=0.7768
[OriginalRS] Epoch 13/30 loss=0.7778
[OriginalRS] Epoch 14/30 loss=0.7738
[OriginalRS] Epoch 15/30 loss=0.7747
[OriginalRS] Epoch 16/30 loss=0.7735
[OriginalRS] Epoch 17/30 loss=0.7752
[OriginalRS] Epoch 18/30 loss=0.7722
[OriginalRS] Epoch 19/30 loss=0.7715
[OriginalRS] Epoch 20/30 loss=0.7707
[OriginalRS] Epoch 21/30 loss=0.7694
[OriginalRS] Epoch 22/30 loss=0.7704
[OriginalRS] Epoch 23/30 loss=0.7709
[OriginalRS] Epoch 24/30 loss=0.7711
[OriginalRS] Epoch 25/30 loss=0.7685
[OriginalRS] Epoch 26/30 los